# Comparing Experiments

This notebook compares results across different experiments.

In [ ]:
from pathlib import Path
import pandas as pd

### Averaging Across Seeds
*Experiments: 10–14*  
This section averages the ResNet-18 candidate experiments, which were run across five different seeds.

In [ ]:
def load_results(label):
    """
    Load results for a given label from the parquet file for experiments 10–14.

    Args:
        label (str): The label for which to load results.

    Returns:
        pd.DataFrame: A DataFrame containing the loaded results.
    """

    label = label + "_classify"
    df = pd.read_parquet(Path("../results") / label / "experiments.parquet")
    df = df[df["experiment_number"].isin([10, 11, 12, 13, 14])]
    return df

In [ ]:
# Columns to average across seeds
METRIC_COLUMNS = [
    "test_acc",
    "test_weighted_f1_avg",
    "test_unseen_matched_acc",
    "test_unseen_matched_weighted_f1_avg",
]


def average_seed_results(results):
    """
    Return mean and standard deviation for each candidate across seeds, rounded to 2 DP.

    Args:
        results (pd.DataFrame): A DataFrame containing the results to be averaged.

    Returns:
        pd.DataFrame: A DataFrame containing the mean and standard deviation for each
            candidate across seeds.
    """

    seed_counts = results.groupby("run_name")["seed"].nunique()
    summary = results.groupby("run_name")[METRIC_COLUMNS].agg(["mean", "std"])
    summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
    summary.insert(0, "n_seeds", seed_counts)
    return summary.reset_index().round(2)


def export_seed_comparison(label):
    """
    Average seed results, export them to CSV, and return the summary.

    Args:
        label (str): The label for which to export seed comparison.

    Returns:
        pd.DataFrame: A DataFrame containing the averaged seed results.
    """

    summary = average_seed_results(load_results(label))
    filename = "candidate_seed_comparison"

    output_dir = Path("../acc_f1_tables") / f"{label}_classify"
    output_dir.mkdir(parents=True, exist_ok=True)
    summary.to_csv(output_dir / f"{filename}.csv", index=False)
    return summary

#### Object

In [ ]:
export_seed_comparison("object")

#### Object Region

In [ ]:
export_seed_comparison("object_region")